In [1]:
# Running the full cohort building pipeline and checking the result.
import sys
sys.path.append("..")

from src.build_cohort import run

cohort = run("../data/companies_snapshot.csv", "../data/cohort.parquet")
cohort.head()

saved 3000 companies to ../data/cohort.parquet


,CompanyName,CompanyNumber,CompanyCategory,CompanyStatus,IncorporationDate,SICCode.SicText_1,Accounts.NextDueDate,Accounts.LastMadeUpDate,Accounts.AccountCategory,ConfStmtNextDueDate,ConfStmtLastMadeUpDate,Mortgages.NumMortCharges,is_failed
0,ALLIANCE CARE LTD,09977882,Private Limited Company,Liquidation,29/01/2016,86900 - Other human health activities,31/03/2025,30/06/2023,MICRO ENTITY,07/11/2024,24/10/2023,2,1
1,BOARDRIDERS UK LTD,04544271,Private Limited Company,Liquidation,24/09/2002,47510 - Retail sale of textiles in specialised...,31/10/2020,31/10/2018,FULL,13/12/2020,01/11/2019,5,1
2,AGMAN COCOA LIMITED,01287947,Private Limited Company,Liquidation,25/11/1976,99999 - Dormant Company,30/06/2026,30/09/2024,AUDIT EXEMPTION SUBSIDIARY,14/06/2026,31/05/2025,1,1
3,ADMIRAL CARE LTD,07161144,Private Limited Company,Liquidation,17/02/2010,86900 - Other human health activities,31/12/2026,31/03/2025,MICRO ENTITY,03/03/2026,17/02/2025,0,1
4,BAB ELECTRICAL SERVICES LIMITED,07682840,Private Limited Company,Liquidation,27/06/2011,43210 - Electrical installation,31/12/2022,31/03/2021,UNAUDITED ABRIDGED,09/08/2022,26/07/2021,0,1


In [2]:
# Confirming the cohort has both groups and the right total size
print(cohort.shape)
cohort["is_failed"].value_counts()

(3000, 13)


is_failed
1    1500
0    1500
Name: count, dtype: int64

In [10]:
# Confirm the API key works before starting the full collection.
import os
import requests
from dotenv import load_dotenv

load_dotenv("../.env")
api_key = os.getenv("CH_API_KEY")
response = requests.get(
    "https://api.company-information.service.gov.uk/company/00006400",
    auth=(api_key, "")
)
print(response.status_code)

200


In [9]:
# Test the collector on just 5 companies before running the full 3000.
import sys
sys.path.append("..")
from src.collect import collect_cohort
import pandas as pd

small = pd.read_parquet("../data/cohort.parquet").head(5)
small.to_parquet("../data/test_cohort.parquet")
collect_cohort("../data/test_cohort.parquet")

finished. collected: 0, skipped: 5, failed: 0


In [15]:
import importlib
import src.collect
importlib.reload(src.collect)
from src.collect import collect_cohort

In [16]:
# Collect filing history and officers for every company in the cohort.
collect_cohort("../data/cohort.parquet")

1550/3000 processed, 62 collected, 1488 skipped, 0 failed
2200/3000 processed, 112 collected, 2088 skipped, 0 failed
2650/3000 processed, 162 collected, 2488 skipped, 0 failed
finished. collected: 180, skipped: 2820, failed: 0


In [12]:
# Check what error we're actually getting on a failing company, to diagnose the real cause.
import requests, os
from dotenv import load_dotenv
load_dotenv("../.env")
api_key = os.getenv("CH_API_KEY")

r = requests.get("https://api.company-information.service.gov.uk/company/00000006", auth=(api_key, ""))
print(r.status_code, r.headers.get("Retry-After"), r.text[:200])

200 None {"accounts":{"accounting_reference_date":{"day":"31","month":"12"},"last_accounts":{"made_up_to":"2014-12-31","period_end_on":"2014-12-31","type":"full"}},"can_file":false,"company_name":"MARINE AND G


In [ ]:
# Confirming the real collection saved usable data by checking one file.
import json
files = list(Path("../data/raw").glob("*.json"))
print(files[0])
data = json.loads(files[0].read_text())
print(data["profile"]["company_name"], data["profile"]["company_status"])
print("filings returned:", len(data["filings"].get("items", [])))
print("officers returned:", len(data["officers"].get("items", [])))

../data/raw/07161144.json
ADMIRAL CARE LTD liquidation
filings returned: 25
officers returned: 7
